In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset=pd.read_csv("insurance_pre.csv")

In [3]:
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
dataset = pd.get_dummies(dataset,dtype=int,drop_first='True')

In [5]:
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [6]:
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [7]:
independant= dataset[['age', 'bmi', 'children','sex_male', 'smoker_yes']]

In [8]:
independant

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [9]:
dependant=dataset[['charges']]
dependant

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [10]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(independant, dependant, test_size=0.30, random_state=0) 

In [11]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

param_grid = {'criterion': ['squared_error', 'absolute_error', 'friedman_mse'],
              'max_features': [None, 'sqrt', 'log2'],
              'splitter': ['best', 'random']}

grid = GridSearchCV(estimator=DecisionTreeRegressor(),param_grid=param_grid,cv=5,
                    n_jobs=-1,verbose=3)

#fitting the model under the grid search
grid.fit(X_train,Y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


,estimator,DecisionTreeRegressor()
,param_grid,"{'criterion': ['squared_error', 'absolute_error', ...], 'max_features': [None, 'sqrt', ...], 'splitter': ['best', 'random']}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'friedman_mse'


In [16]:
GridSearchCV(cv='5', error_score='raise',
             estimator=DecisionTreeRegressor(criterion='squared_error', max_depth=None,
                                             max_features=None,
                                             max_leaf_nodes=None,
                                             min_impurity_decrease=0.0,
                                             min_samples_leaf=1,
                                             min_samples_split=2,
                                             min_weight_fraction_leaf=0.0,
                                             random_state=None,
                                             splitter='best'), n_jobs=-1,
             param_grid={'criterion': ['squared_error', 'absolute_error', 'friedman_mse'],'max_features': [None, 'sqrt', 'log2'],'splitter': ['best', 'random']},
             pre_dispatch='2*n_jobs', refit=True, return_train_score=False,scoring=None, verbose=3)

,estimator,DecisionTreeRegressor()
,param_grid,"{'criterion': ['squared_error', 'absolute_error', ...], 'max_features': [None, 'sqrt', ...], 'splitter': ['best', 'random']}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,'5'
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,'raise'
,return_train_score,False
,criterion,'squared_error'


In [19]:
# print best parameter after tuning
#print(grid.best_params_)
re=grid.cv_results_#print(re)
grid_predictions = grid.predict(X_test)

# print classification report
from sklearn.metrics import r2_score
r_score=r2_score(Y_test,grid_predictions)

print("The R_score value for best parameter {}:".format(grid.best_params_),r_score)

The R_score value for best parameter {'criterion': 'friedman_mse', 'max_features': None, 'splitter': 'best'}: 0.6956516288573451


In [20]:
table=pd.DataFrame.from_dict(re)

In [21]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.010109,0.004008,0.004363,0.001320,squared_error,None,best,"{'criterion': 'squared_error', 'max_features':...",0.730916,0.540929,0.778768,0.632527,0.680028,0.672633,0.082062,3
1,0.012315,0.001407,0.007086,0.001217,squared_error,None,random,"{'criterion': 'squared_error', 'max_features':...",0.541735,0.710032,0.633182,0.605832,0.625949,0.623346,0.054009,8
2,0.011446,0.001564,0.005901,0.001055,squared_error,sqrt,best,"{'criterion': 'squared_error', 'max_features':...",0.748806,0.508485,0.456065,0.693506,0.530792,0.587531,0.113129,13
3,0.009060,0.000778,0.005228,0.000737,squared_error,sqrt,random,"{'criterion': 'squared_error', 'max_features':...",0.633686,0.543280,0.566160,0.425363,0.627917,0.559281,0.075478,17
4,0.010044,0.001292,0.005545,0.001410,squared_error,log2,best,"{'criterion': 'squared_error', 'max_features':...",0.728132,0.626658,0.542683,0.709184,0.420815,0.605494,0.113394,11
5,0.009453,0.001290,0.005753,0.001073,squared_error,log2,random,"{'criterion': 'squared_error', 'max_features':...",0.600864,0.576024,0.483750,0.610562,0.587703,0.571781,0.045543,15
6,0.044413,0.003096,0.003930,0.000559,absolute_error,None,best,"{'criterion': 'absolute_error', 'max_features'...",0.743783,0.585949,0.612209,0.546613,0.647693,0.627250,0.066981,6
7,0.028179,0.001744,0.003464,0.000632,absolute_error,None,random,"{'criterion': 'absolute_error', 'max_features'...",0.728340,0.721397,0.631853,0.655979,0.631787,0.673871,0.042620,2
8,0.022630,0.002890,0.003003,0.000312,absolute_error,sqrt,best,"{'criterion': 'absolute_error', 'max_features'...",0.570130,0.379122,0.683392,0.263295,0.631583,0.505504,0.158968,18
9,0.015260,0.000695,0.003457,0.000753,absolute_error,sqrt,random,"{'criterion': 'absolute_error', 'max_features'...",0.788630,0.571810,0.635356,0.461192,0.635088,0.618415,0.106235,9


In [22]:
age_input=float(input("Age:"))
bmi_input=float(input("BMI:"))
children_input=float(input("Children:"))
sex_male_input=int(input("Sex Male 0 or 1:"))
smoker_yes_input=int(input("Smoker Yes 0 or 1:"))

Age: 43
BMI: 32
Children: 2
Sex Male 0 or 1: 1
Smoker Yes 0 or 1: 0


In [23]:
Future_Prediction=grid.predict([[age_input,bmi_input,children_input,sex_male_input,smoker_yes_input]])
#change the paramter,play with it.
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[16085.1275]
